In [2]:
! pip install biopython
! pip install -q condacolab
import condacolab
condacolab.install()
! conda install -c bioconda seqkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.0 MB/s eta 0:00:00
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...
Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: \ | / - done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - seqkit


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    certifi-2026.2.25          |     pyhd8ed1ab_0         148 KB  conda-forge
    conda-24.11.3              |  py311h38be061_0         1.1 MB  conda-forge
    openssl-3.6.1              |       h35e630c_1         3.0 MB  conda-forge
    seqkit-2.13.0     

In [81]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys

class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name

    @staticmethod
    def _get_uniprot(accession):
        '''
        define http_function to get the data from Uniprot API
        '''
        endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
        http_function = requests.get
        http_args = {'params': {'accessions': accession}}
        return http_function(endpoint, **http_args)

    @staticmethod
    def _uniprot_parse_response(resp: dict):
        '''
        parse response from Uniprot and output
        organism, geneInfo, sequenceInfo, type

        do not forget to include error handling (e.g. for key errors)
        '''
        try:
            data = resp.json()

            entry = data['results'][0]
            return {
                'organism': entry.get('organism', {}).get('scientificName'),
                'geneInfo': entry.get('genes'),
                'sequenceInfo': entry.get('sequence'),
                'type': entry.get('entryType')
                }
        except Exception as e:
            return e

    @staticmethod
    def _get_ensembl(id):
        '''
        define http_function to get the data from Ensembl API
        '''
        endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
        http_function = requests.get
        http_args = {"headers": {"Content-Type": "application/json"}}
        return http_function(endpoint, **http_args)

    @staticmethod
    def _ensembl_parse_response(resp: dict):
      '''
      parse Ensembl response and output
      object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source

      do not forget to include error handling (e.g. for key errors)
      '''
      try:
        data = resp.json()
        fields = [
            'object_type', 'species', 'assembly_name', 'biotype',
            'display_name', 'id', 'db_type', 'description', 'source',
            'canonical_transcript'
        ]
        return {field: data.get(field) for field in fields}
      except Exception as e:
        return e

    def _access_database(self, identifier, db_type, seq_description, seq_sequence) -> dict:
        # this function calls either _get_uniprot() or _get_ensembl()
        # an then _uniprot_parse_response() or _ensembl_parse_response()
        # outputs a dictionary with results (see example in test_data)

        uniprot_pattern = r'[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}'
        ensembl_pattern = r'ENS[A-Z]+[0-9]{11}'

        if db_type.lower() == "protein":
            pattern = uniprot_pattern
            get_func = self._get_uniprot
            parse_func = self._uniprot_parse_response
        else:
            pattern = ensembl_pattern
            get_func = self._get_ensembl
            parse_func = self._ensembl_parse_response

        match = re.search(pattern, identifier)
        if not match:
            return {"WARNING": {"No ID match found."}}

        acsession = match.group(0)
        response = get_func(acsession)
        db_info = parse_func(response)

        if not db_info:
            return {"WARNING": {"No ID match found."}}

        return {
            f"file_info_{acsession}": {
                "description": seq_description,
                "sequence": seq_sequence
            },
            f"database_info_{acsession}": db_info
        }


    def seqkit_stats(self) -> dict:
        # this function calls seqkit via subprocess
        # if error arises, returns stderr and finishes the parser execution
        # if stats are collected, return the result (see example in test_data)
        try:
            process = subprocess.run(["seqkit", "stats", self.filename, "-a", "-T"],
                                     capture_output=True, text=True, check=True)

            lines = process.stdout.strip().split('\n')

            headers = lines[0].split('\t')
            values = lines[1].split('\t')
            return dict(zip(headers, values))

        except Exception as e:
            return {"error":  'Seqkit reading error:' + str(e)}

    def biopython_parser(self, seqkit_result) -> dict:
        # accepts seqkit_result dict
        # depending on fasta file type selects regular expression
        # parses fasta file via biopython
        # iterates over sequences and calls relevant database via _access_databases()
        # returns final result (info about each sequence from FASTA file + info from database)

        if "error" in seqkit_result:
            return seqkit_result

        file_type = seqkit_result.get('type', '')
        db_name = "uniprot" if file_type.lower() == "protein" else "ensembl"

        output = {"DB_name": db_name}

        try:
            for rec in SeqIO.parse(self.filename, "fasta"):
                upd = self._access_database(
                    rec.description,
                    file_type,
                    rec.description,
                    str(rec.seq))
                output.update(upd)

            return output
        except Exception as e:
            return {"error": str(e)}

    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [82]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print(stats)
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'file': 'test_file.fasta', 'format': 'FASTA', 'type': 'Protein', 'num_seqs': '2', 'sum_len': '456', 'min_len': '29', 'avg_len': '228.0', 'max_len': '427', 'Q1': '29', 'Q2': '228', 'Q3': '427', 'sum_gap': '0', 'N50': '427', 'N50_num': '1', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0.00', 'GC(%)': '0.00', 'sum_n': '0'}
DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0

In [83]:
parser = MyFastaParser('ensembl_download_1.fasta')
stats = parser.seqkit_stats()
print(stats)
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'file': 'ensembl_download_1.fasta', 'format': 'FASTA', 'type': 'DNA', 'num_seqs': '6', 'sum_len': '86', 'min_len': '9', 'avg_len': '14.3', 'max_len': '23', 'Q1': '10', 'Q2': '14', 'Q3': '17', 'sum_gap': '0', 'N50': '16', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0.00', 'GC(%)': '45.35', 'sum_n': '0'}
DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221
	object_type
		Transcript
	species
		mus_musculus
	assembly_name
		GRCm39
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST00000196221
	db_type
		core
	description
		None
	source
		havana
	canonical_transcript
		None
file_info_ENSMUST00000177564
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:543

In [84]:
parser = MyFastaParser('ensembl_download_2.fasta')
stats = parser.seqkit_stats()
print(stats)
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'error': 'Seqkit reading error:list index out of range'}
error
	Seqkit reading error:list index out of range


In [85]:
parser = MyFastaParser('uniprot_download.fasta')
stats = parser.seqkit_stats()
print(stats)
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

{'file': 'uniprot_download.fasta', 'format': 'FASTA', 'type': 'Protein', 'num_seqs': '7', 'sum_len': '3861', 'min_len': '180', 'avg_len': '551.6', 'max_len': '1382', 'Q1': '429', 'Q2': '441', 'Q3': '500', 'sum_gap': '0', 'N50': '468', 'N50_num': '3', 'Q20(%)': '0', 'Q30(%)': '0', 'AvgQual': '0.00', 'GC(%)': '0.00', 'sum_n': '0'}
DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
